In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [2]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Mirai\\combined_mirai.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "mirai_ann.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\Mirai\combined_mirai.csv | Total rows: 100000
Train: 60000 | Val: 20000 | Test: 20000


In [3]:
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.FloatTensor(y_train.values.copy())
X_val_t   = torch.FloatTensor(X_val_scaled)
y_val_t   = torch.FloatTensor(y_val.values.copy())
X_test_t  = torch.FloatTensor(X_test_scaled)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

In [4]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]

class ANNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model     = ANNModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

In [5]:
epochs = 50
patience          = 5
best_val_loss     = float("inf")
epochs_no_improve = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t.to(device))
        val_loss  = criterion(val_preds, y_val_t.to(device)).item()
        val_acc   = ((val_preds > 0.5) == y_val_t.to(device)).float().mean().item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(MODEL_PATH))

Epoch 1/50 | Loss: 0.5541 | Val Loss: 0.4991 | Val Acc: 0.7286
Epoch 2/50 | Loss: 0.5072 | Val Loss: 0.4808 | Val Acc: 0.7527
Epoch 3/50 | Loss: 0.4874 | Val Loss: 0.4646 | Val Acc: 0.7652
Epoch 4/50 | Loss: 0.4714 | Val Loss: 0.4568 | Val Acc: 0.7947
Epoch 5/50 | Loss: 0.4608 | Val Loss: 0.4430 | Val Acc: 0.7493
Epoch 6/50 | Loss: 0.4541 | Val Loss: 0.4260 | Val Acc: 0.8070
Epoch 7/50 | Loss: 0.4433 | Val Loss: 0.4439 | Val Acc: 0.7605
Epoch 8/50 | Loss: 0.4398 | Val Loss: 0.4199 | Val Acc: 0.8105
Epoch 9/50 | Loss: 0.4256 | Val Loss: 0.4240 | Val Acc: 0.7967
Epoch 10/50 | Loss: 0.4163 | Val Loss: 0.4007 | Val Acc: 0.8073
Epoch 11/50 | Loss: 0.4138 | Val Loss: 0.3902 | Val Acc: 0.8170
Epoch 12/50 | Loss: 0.4114 | Val Loss: 0.3901 | Val Acc: 0.8177
Epoch 13/50 | Loss: 0.4083 | Val Loss: 0.3865 | Val Acc: 0.8180
Epoch 14/50 | Loss: 0.4064 | Val Loss: 0.4056 | Val Acc: 0.8108
Epoch 15/50 | Loss: 0.4062 | Val Loss: 0.3853 | Val Acc: 0.8204
Epoch 16/50 | Loss: 0.4025 | Val Loss: 0.3753 | V

<All keys matched successfully>

In [6]:
model.eval()
with torch.no_grad():
    preds = (model(X_test_t.to(device)) > 0.5).cpu().numpy().astype(int)

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.83      0.90      0.86     10000
      Attack       0.89      0.81      0.85     10000

    accuracy                           0.86     20000
   macro avg       0.86      0.86      0.86     20000
weighted avg       0.86      0.86      0.86     20000

[[9001  999]
 [1876 8124]]
